# مرحله ۳ — یادگیری ماشین اعتبارسنجی با بهینه‌سازی ماتریس هزینه
**نام پروژه:** سامانه ارزیابی ریسک اعتباری متقاضیان وام (Give Me Some Credit)  
**محیط اجرا:** Python 3.12 | PostgreSQL 16 | SQLAlchemy 2.0 | scikit-learn  
**پایگاه داده مبدأ:** `credit.vw_borrowers_clean` (PostgreSQL)  
**ماتریس هزینه کسب‌وکار:** جریمه نکول کشف‌نشده (False Negative) = ۱۲ واحد | جریمه رد اشتباه مشتری سالم (False Positive) = ۱ واحد

---

## ۱. نقشه راه و اهداف این نوت‌بوک
این نوت‌بوک الزامات چالش مرحله ۳ را بر اساس راهنمای تحویل (`00_Delivery_Guide`) برآورده می‌سازد:
1. **عدم خواندن مستقیم CSV**: تمام داده‌ها از پایگاه داده PostgreSQL و از طریق نمای `credit.vw_borrowers_clean` با کتابخانه SQLAlchemy خوانده می‌شوند.
2. **شناخت ساختار داده و برچسب**: بررسی اندازه نمونه‌ها، نوع ویژگی‌ها، و عدم توازن شدید کلاس هدف (~۶.۶۸٪ نکول).
3. **تحلیل عمیق مقادیر نگهبان (Sentinel Values)**: شناسایی کدهای اعتباری ۹۶ و ۹۸، همبستگی کامل میان ستون‌های تأخیر، بررسی نرخ نکول فوق‌العاده بالا در این افراد (۵۴.۶٪) و اتخاذ استراتژی پالایش و کدگذاری دودویی.
4. **بررسی داده‌های پرت غیرممکن (Impossible Outliers)**: سن صفر سال و بدهی‌های ذخیره‌شده به عنوان مبالغ دلاری.


In [ ]:
# تنظیمات اولیه، بذر تصادفی ثابت و واردسازی کتابخانه‌ها
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sqlalchemy import text

# افزودن مسیر پکیج به مسیرهای پایتون
sys.path.insert(0, str(Path("src").resolve()))

import credit_risk
from credit_risk.cost import (
    DEFAULT_COST_FN,
    DEFAULT_COST_FP,
    calculate_expected_cost_per_thousand,
    calculate_total_cost,
    find_optimal_threshold,
)
from credit_risk.db import get_db_engine, load_clean_borrowers, load_raw_borrowers
from credit_risk.features import add_engineered_features
from credit_risk.sentinel import SentinelAuditor, clean_sentinel_values

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
print("کتابخانه‌ها با موفقیت بارگذاری شدند. بذر تصادفی:", RANDOM_SEED)


کتابخانه‌ها با موفقیت بارگذاری شدند. بذر تصادفی: 42


---
## ۲. استخراج داده از پایگاه داده (الزام زنجیره داده)
> **قاعده سخت:** داده مستقیماً از فایل CSV خوانده نمی‌شود، بلکه داده خام توسط کارخانه داده اعتبارسنجی شده، در PostgreSQL بارگذاری شده، و اکنون از نمای تحلیلی `credit.vw_borrowers_clean` خوانده می‌شود.


In [2]:
# اتصال به PostgreSQL و واکشی نمای آماده‌سازی شده
engine = get_db_engine()

# اجرای کوئری با SQLAlchemy
with engine.connect() as conn:
    db_name = conn.execute(text("SELECT current_database();")).scalar()
    tbl_count = conn.execute(text("SELECT count(*) FROM credit.borrowers;")).scalar()
    view_count = conn.execute(text("SELECT count(*) FROM credit.vw_borrowers_clean;")).scalar()

print(f"پایگاه داده متصل: {db_name}")
print(f"تعداد رکوردهای جدول اصلی (credit.borrowers): {tbl_count:,}")
print(f"تعداد رکوردهای نمای تحلیلی (credit.vw_borrowers_clean): {view_count:,}")

# واکشی کامل دیتافریم از نمای پایگاه داده
df = load_clean_borrowers(engine=engine)
print(f"ابعاد داده‌های واکشی شده در پایتون: {df.shape}")
df.head(5)


پایگاه داده متصل: olist
تعداد رکوردهای جدول اصلی (credit.borrowers): 150,000
تعداد رکوردهای نمای تحلیلی (credit.vw_borrowers_clean): 150,000


ابعاد داده‌های واکشی شده در پایتون: (150000, 22)


,applicant_id,serious_dlqin_2yrs,revolving_utilization_of_unsecured_lines,age,has_sentinel_late_code,clean_num_30_59_days_late,clean_num_60_89_days_late,clean_num_90_days_late,clean_total_past_due_events,is_monthly_income_missing,...,is_debt_ratio_as_dollar_amount,debt_ratio,monthly_income,number_of_open_credit_lines_and_loans,number_real_estate_loans_or_lines,number_of_dependents,raw_num_30_59_days_late,raw_num_60_89_days_late,raw_num_90_days_late,ingested_at_utc
0,1,1,0.766127,45,False,2.0,0.0,0.0,2.0,False,...,False,0.802982,9120.0,13,6,2.0,2,0,0,2026-11-04 14:21:45.187601+00:00
1,2,0,0.957151,40,False,0.0,0.0,0.0,0.0,False,...,False,0.121876,2600.0,4,0,1.0,0,0,0,2026-11-04 14:21:45.187601+00:00
2,3,0,0.658180,38,False,1.0,0.0,1.0,2.0,False,...,False,0.085113,3042.0,2,0,0.0,1,0,1,2026-11-04 14:21:45.187601+00:00
3,4,0,0.233810,30,False,0.0,0.0,0.0,0.0,False,...,False,0.036050,3300.0,5,0,0.0,0,0,0,2026-11-04 14:21:45.187601+00:00
4,5,0,0.907239,49,False,1.0,0.0,0.0,1.0,False,...,False,0.024926,63588.0,7,1,0.0,1,0,0,2026-11-04 14:21:45.187601+00:00


---
## ۳. شناخت داده و عدم توازن برچسب هدف (Target Label Analysis)
مؤسسه اعتباری می‌خواهد پیش از اعطای وام احتمال نکول متقاضی (`serious_dlqin_2yrs`) را برآورد کند.
- کلاس ۰: مشتری خوش‌حساب (سالم)
- کلاس ۱: مشتری با تأخیر جدی ۹۰ روز یا بیشتر طی ۲ سال آینده (نکول)


In [3]:
# بررسی توزیع برچسب هدف
counts = df["serious_dlqin_2yrs"].value_counts()
ratios = df["serious_dlqin_2yrs"].value_counts(normalize=True) * 100

summary_target = pd.DataFrame({
    "عنوان کلاس": ["خوش‌حساب (سالم)", "نکول جدی (پرخطر)"],
    "کد": [0, 1],
    "تعداد": [int(counts.get(0, 0)), int(counts.get(1, 0))],
    "درصد": [f"{ratios.get(0, 0):.2f}%", f"{ratios.get(1, 0):.2f}%"],
})
print("توزیع کلاس هدف در کل جامعه آماری:")
print(summary_target.to_string(index=False))

# ماتریس هزینه کسب‌وکار
print("\nماتریس هزینه تعریف‌شده برای مسئله:")
print(f"- هزینه نکول کشف‌نشده (False Negative): {DEFAULT_COST_FN} واحد")
print(f"- هزینه رد اشتباه مشتری سالم (False Positive): {DEFAULT_COST_FP} واحد")


توزیع کلاس هدف در کل جامعه آماری:
      عنوان کلاس  کد  تعداد   درصد
 خوش‌حساب (سالم)   0 139974 93.32%
نکول جدی (پرخطر)   1  10026  6.68%

ماتریس هزینه تعریف‌شده برای مسئله:
- هزینه نکول کشف‌نشده (False Negative): 12.0 واحد
- هزینه رد اشتباه مشتری سالم (False Positive): 1.0 واحد


---
## ۴. بررسی عمیق مقادیر نگهبان (Sentinel Values) و ناهنجاری‌ها
> **الزام چالش:** در این داده بعضی ستون‌ها مقادیری دارند که عدد واقعی نیستند بلکه کدهای نگهبان (Sentinel) هستند. پیدا کردنشان و تعیین تکلیف‌شان بخشی از مسئله است. اگر آن‌ها را عدد معمولی فرض کنید، مدل‌تان روی داده واقعی خراب می‌شود.

### شناسایی کدهای نگهبان:
در سه ستون مربوط به تأخیرهای پرداختی:
- `raw_num_30_59_days_late`
- `raw_num_60_89_days_late`
- `raw_num_90_days_late`

اعدادی مانند **۹۶** و **۹۸** وجود دارند. در استانداردهای سیستم‌های اعتبارسنجی بانکی (مانند Metro 2):
- کد ۹۶ نشان‌دهنده «داده غیرقابل استعلام یا در وضعیت اعتراض/مسدودی» است.
- کد ۹۸ نشان‌دهنده «عدم دسترسی به پیشینه یا خطای سیستمی در استعلام» است.
این افراد ۹۸ بار تأخیر نداشته‌اند!


In [4]:
# ممیزی و بازرسی مقادیر نگهبان با استفاده از ماژول SentinelAuditor
audit_results = SentinelAuditor.audit(df)

print("نتایج بازرسی مقادیر نگهبان:")
print(f"تعداد رکوردهای دارای کد 96 در ستون‌ها: {audit_results['counts_96']}")
print(f"تعداد رکوردهای دارای کد 98 در ستون‌ها: {audit_results['counts_98']}")
print(f"انطباق کامل کد 98 در هر سه ستون: {audit_results['exact_concordance_all_98']} رکورد")
print(f"انطباق کامل کد 96 در هر سه ستون: {audit_results['exact_concordance_all_96']} رکورد")
print(f"مجموع کل رکوردهای دارای کد نگهبان: {audit_results['total_sentinel_rows']} رکورد")
print(f"نرخ نکول در متقاضیان عادی: {audit_results['normal_default_rate']*100:.2f}%")
print(f"نرخ نکول در متقاضیان دارای کد نگهبان: {audit_results['sentinel_default_rate']*100:.2f}%")


نتایج بازرسی مقادیر نگهبان:
تعداد رکوردهای دارای کد 96 در ستون‌ها: {'raw_num_30_59_days_late': 5, 'raw_num_60_89_days_late': 5, 'raw_num_90_days_late': 5}
تعداد رکوردهای دارای کد 98 در ستون‌ها: {'raw_num_30_59_days_late': 264, 'raw_num_60_89_days_late': 264, 'raw_num_90_days_late': 264}
انطباق کامل کد 98 در هر سه ستون: 264 رکورد
انطباق کامل کد 96 در هر سه ستون: 5 رکورد
مجموع کل رکوردهای دارای کد نگهبان: 269 رکورد
نرخ نکول در متقاضیان عادی: 6.60%
نرخ نکول در متقاضیان دارای کد نگهبان: 54.65%


### تحلیل کسب‌وکاری و اثر مقادیر نگهبان:
1. **انطباق صددرصدی:** هر ۲۶۴ متقاضی که در یک ستون کد ۹۸ دارند، در هر دو ستون دیگر نیز دقیقاً کد ۹۸ دارند! ۵ متقاضی کد ۹۶ نیز در هر سه ستون کد ۹۶ دارند (مجموعاً ۲۶۹ متقاضی).
2. **سیگنال ریسک فوق‌العاده قوی:** متقاضیانی که کد نگهبان دارند، نرخ نکول **۵۴.۶۵٪** دارند (در مقایسه با ۶.۶۰٪ در افراد عادی)—یعنی ریسک نکول آن‌ها **بیش از ۸ برابر** میانگین جامعه است!
3. **خطای ریاضی در صورت عدم تفکیک:** اگر عدد ۹۸ به عنوان ۹۸ بار دیرکرد واقعی وارد رگرسیون لجستیک یا شبکه‌های عصبی شود، وزن این متغیرها به دلیل مقیاس بزرگ به شدت منحرف می‌شود.
4. **استراتژی اصولی برخورد:**
   - استخراج متغیر دودویی `has_sentinel_late_code = 1` برای حفظ کامل سیگنال ریسک ۵۴٪.
   - ماسک کردن خود اعداد ۹۶ و ۹۸ به مقدار خالی (NaN) یا صفر، تا توزیع عددی تأخیرهای پرداختی آلوده نشود.


In [5]:
# بررسی داده‌های پرت غیرممکن (Impossible Outliers)
# ۱. سن غیرممکن: سن کمتر از ۱۸ سال
zero_age = df[df["age"] < 18]
print(f"تعداد افراد با سن غیرممکن (< 18 سال): {len(zero_age)}")
if len(zero_age) > 0:
    print(zero_age[["applicant_id", "age", "serious_dlqin_2yrs", "monthly_income", "debt_ratio"]].to_string(index=False))

# ۲. تحلیل وضعیت نسبت بدهی در غیاب درآمد ماهانه
missing_inc = df[df["monthly_income"].isna()]
print(f"تعداد رکوردهای فاقد درآمد ماهانه: {len(missing_inc):,}")
print(f"میانه نسبت بدهی در افراد فاقد درآمد: {missing_inc['debt_ratio'].median():.2f}")
print(f"میانه نسبت بدهی در افراد دارای درآمد: {df[df['monthly_income'].notna()]['debt_ratio'].median():.2f}")


تعداد افراد با سن غیرممکن (< 18 سال): 1
 applicant_id  age  serious_dlqin_2yrs  monthly_income  debt_ratio
        65696    0                   0          6000.0    0.436927
تعداد رکوردهای فاقد درآمد ماهانه: 29,731
میانه نسبت بدهی در افراد فاقد درآمد: 1159.00
میانه نسبت بدهی در افراد دارای درآمد: 0.30


---
## ۵. وضعیت پروژه و آمادگی برای بخش‌های بعدی
تا این مرحله:
- پایگاه داده PostgreSQL و نمای تمیزساز `credit.vw_borrowers_clean` با موفقیت مستقر شدند.
- ۱۵۰٬۰۰۰ رکورد بدون از دست رفتن داده‌های ناقص بارگذاری شدند.
- مقادیر نگهبان (Sentinel Values) به صورت دقیق شناسایی شده و سیگنال ریسک ۵۴.۶٪ آن‌ها کشف گردید.
- متغیرهای دودویی نگهبان در لایه دیتابیس پیاده‌سازی شدند.

**گام‌های بعدی چالش:**
- بررسی الگوهای گم‌شدگی (MCAR / MAR) و مقایسه دو سیاست انتساب (Imputation).
- مهندسی ۴ ویژگی جدید همراه با ممیزی نشت اطلاعات (Leakage Audit).
- آموزش مدل‌های خانواده رگرسیون لجستیک و درخت تصمیم/گرادیان بوستینگ.
- بهینه‌سازی آستانه تصمیم‌گیری بر اساس ماتریس هزینه ۱۲:۱ و گزارش هزینه به ازای هر ۱۰۰۰ متقاضی.
